# 30. 구조화 DB 생성·보존 점검

OCR 통과 가정. 원본은 읽기 전용이며 현재 Upstage 계층을 수정하지 않습니다. Python 실행은 conda skn25만 허용합니다.

## 재현과 책임

장시간 API·모델 실행은 체크포인트·동일 payload 캐시·실패 기록을 갖춘 아래 runner가 담당합니다. 이 노트북은 실제 저장 결과를 읽고 검증·표시하고 연결 보고서를 생성합니다. 결과를 만들어 넣지 않습니다.

각 명령은 `PYTHONDONTWRITEBYTECODE=1 conda run --no-capture-output -n skn25`로 실행합니다. DB 쓰기는 database-manager가 담당하며 운영 publication runner는 실행하지 않습니다.

```bash
python -B notebooks/30_build_structured_db.py
```

In [1]:
from pathlib import Path
import importlib.util, sys, json
root=Path.cwd()
if root.name=='notebooks': root=root.parent
spec=importlib.util.spec_from_file_location('structured_common',root/'notebooks/30_structured_search_common.py')
C=importlib.util.module_from_spec(spec);sys.modules[spec.name]=C;spec.loader.exec_module(C)
assert Path(sys.prefix).name=='skn25'
print({'environment':'skn25','model':C.MODEL,'reasoning':C.REASONING,'originals':'read only'})

{'environment': 'skn25', 'model': 'gpt-6-luna', 'reasoning': 'medium', 'originals': 'read only'}


In [2]:
files=['db_build_summary.json']
for name in files:
    path=C.STAGES[30]/name
    if not path.exists(): raise RuntimeError('Required stage artifact not complete: '+name)
    data=C.read_json(path)
    if 'records' in data:
        print(name,{'records':len(data['records']),'status_counts':{s:sum(r['status']==s for r in data['records']) for s in ['ok','failed']}})
    elif 'queries' in data: print(name,{'queries':len(data['queries']),'sha256':data['sha256']})
    else: print(json.dumps(data,ensure_ascii=False,indent=2)[:15000])

{
  "artifact_sha256": {
    "documents.jsonl": "fdf855a8d3b11103f776f913b010860c6b9c5e53af5b63a61738683c3d678eac",
    "facts.jsonl": "bdd116338687101a0362c703839ca24056abbe0443c50b7bd622f905db1b3d9c",
    "structured.sqlite": "65cb90fd3d41ab8b9d3ba3db1cc7e841d2415768f278f0e7d3337c96fb9a38c8"
  },
  "counts": {
    "documents": 106,
    "facts": 3403,
    "known_cap": 381,
    "known_cap_period": 66,
    "known_percent": 1033,
    "known_spend_threshold": 187,
    "pages": 617,
    "positive_supported_action": 1818
  },
  "limitations": [
    "OCR and normalization correctness were not revalidated.",
    "Derived numbers remain unknown unless typed and raw values uniquely agree.",
    "Percent ranges, bounds, and mixed money/percent expressions are not treated as guaranteed rates.",
    "Spend thresholds require an explicit previous-month phrase and a connected single minimum.",
    "Monthly and daily caps only match when the same fact has one explicit period; absent or mixed periods 

## 해석 범위

source-relative raw OCR 필드 라벨과 새로운 개발/확증 cohort입니다. 독립 PDF 검증, 전수 정답 recall, 운영 채택을 뜻하지 않습니다. 프로필 비교는 청킹·재정렬·근거 구성 전체의 비교입니다. 완전한 재현은 입력 hash·원본 코드 hash·새 인덱스·저장 API 응답과 보고서의 실패 분모를 함께 확인해야 합니다.

In [3]:
spec=importlib.util.spec_from_file_location('experiment_reports',root/'notebooks/30_render_experiment_reports.py')
R=importlib.util.module_from_spec(spec);sys.modules[spec.name]=R;spec.loader.exec_module(R)
print(R.render(30).relative_to(root))

docs/report/30_structured_benefit_db_report.html
